# Complete example

Taken from Klibi et al chapters 7, 10 and 11. We use their supply chain network. For this example, we assume short-term design and exclude the evolutionary paths mentioned in chapter 10: we assume demand follows a single baseline distribution, hence it is fixed to benchmark levels so that any network reconfiguration will be solely driven by hazard disruptions. 

For the given example, this notebook constructs the end-to-end pipeline, from hazard simulation to optimization.

## Configuration and set-up

In [ ]:
import pulp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json 
import ast

SEED = 123
np.random.seed(SEED)

# hazard simulation function
from monte_carlo import generate_mc_scenario_only, is_warehouse_hit

# downtime & recovery functions
from rosenheim_TTR import compute_scenario_capacity, evaluate_proxies_continuous_Klibi
from single_echelon_stochastic import build_and_solve_one_stage

## Network topology and benchmark parameters

The commented values for alpha ($0.30$, $0.20$ and $0.40$ for `Germany`, `France`, `Italy` respectively, were used in the beginning of the project, yielding the results presented during the midterm presentation). The new values result from our computations, and can be found in `attenuation_probas.csv`. 

In [2]:
# benchmark data from chapter 11
json_path = "../Data/Dummy_examples/dummySN_chap11.json"
with open(json_path, "r") as f :
    data_SN = json.load(f)
candidate_dcs = data_SN["candidate_dcs"]
demand_zones = data_SN["demand_zones"]

periods = [1, 2, 3] # 3 years planning horizon

CANDIDATE_WAREHOUSES = {
    "Dusseldorf": {
        "country": "Germany",
        "alpha": 0.1950203114176292, # 0.30,
        "nominal_cap": data_SN["nominal_capacities"]["Dusseldorf"],
    },
    "Toulouse": {
        "country": "France",
        "alpha": 0.1784314270696093, # 0.20,
        "nominal_cap": data_SN["nominal_capacities"]["Toulouse"],
    },
    "Turin": {
        "country": "Italy",
        "alpha": 0.2017065804625539, # 0.40,
        "nominal_cap": data_SN["nominal_capacities"]["Turin"],
    },
}

flow_costs = {
    (w, l): data_SN["flow_costs"][w][l]
    for w in candidate_dcs
    for l in demand_zones
}
min_deliveries = {
    (l, int(t)): val
    for t, zones in data_SN["min_deliveries"].items()
    if int(t) in periods
    for l, val in zones.items()
}

In [3]:
# Monte Carlo inputs
df_raw_events = df_raw_events = pd.read_csv("../Data/Results/CLEAN_disasters_europe.csv")

# because sometimes the country names recorded are in tuples
def clean_country_name(country) :

    if not isinstance(country, str) :
        return str(country)
    s = country.strip().strip("'\"")
    
    if country.startswith("(") and country.endswith(")") :
        try :
            parsed = ast.literal_eval(country)
            if isinstance(parsed, tuple) and len(parsed) > 0:
                return str(parsed[0]).strip()
        except Exception:
            pass

    for c in [",", ")", "(", "'", '"'] :
        s = s.replace(c, "")
    return s.strip()

df_raw_events["country_clean"] = df_raw_events["Country"].apply(clean_country_name)

In [4]:
N_EUROPE_EVENTS = len(df_raw_events)
COUNTRY_WEIGHTS = (df_raw_events.groupby("country_clean")["DisNo."].count() / N_EUROPE_EVENTS).to_dict()

DURATIONS = df_raw_events.groupby("country_clean")["event_duration"].apply(list).to_dict()
DURATIONS["Fallback"] = df_raw_events["event_duration"].dropna().tolist()

print(f"Loaded {N_EUROPE_EVENTS} events across {len(COUNTRY_WEIGHTS)} countries.")

Loaded 1694 events across 40 countries.


## Disruption scenarios generation (Monte Carlo)

In [5]:
# Calibrated values from EMDAT fit
NB_N_MONTHLY = 1.4
NB_P_MONTHLY = 0.264

ALPHA = 0.25
DAYS_PER_YEAR = 365
N_YEARS = 3 # planning horizon
N_SCENARIOS = 100

df_event_scenarios = generate_mc_scenario_only(
    nb_n_monthly = NB_N_MONTHLY,
    n_years = N_YEARS,
    nb_p = NB_P_MONTHLY,
    country_weights = COUNTRY_WEIGHTS,
    duration_pools = DURATIONS,
    num_scenarios = N_SCENARIOS,
    severity_function = "severity_proxy",
    seed = SEED
)

df_facility_hits = is_warehouse_hit(CANDIDATE_WAREHOUSES, df_event_scenarios, default_alpha = ALPHA)
df_actual_hits = df_facility_hits[df_facility_hits["is_hit"] == True].copy()

Now we translate scenario severity to closed days.

In [6]:
# translate 
df_actual_hits["days_closed"] = df_actual_hits.apply(
    lambda r: evaluate_proxies_continuous_Klibi(r["duration_days"], r["severity"], facility_type="depot")["days_closed_ttr"],
    axis=1
)

print(f"Generated {len(df_actual_hits)} direct warehouse hits across scenarios.")

Generated 653 direct warehouse hits across scenarios.


## Aggregate downtime & scenario capacities

In [7]:
# First we aggregate downtime per scenarion and warehouse.
df_scenario_impact = df_actual_hits.groupby(["scenario_id", "year", "warehouse"]).agg(
    total_days_closed = ("days_closed", "sum"),
    hit_count = ("is_hit", "sum"),
).reset_index()

downtime_map = df_scenario_impact.set_index(["scenario_id", "year", "warehouse"])["total_days_closed"].to_dict()

df_scenario_impact.head()

,scenario_id,year,warehouse,total_days_closed,hit_count
0,s_1,1,Turin,22.07,1
1,s_1,2,Toulouse,14.27,1
2,s_1,3,Dusseldorf,72.34,1
3,s_1,3,Toulouse,98.57,2
4,s_10,1,Dusseldorf,11.49,1


The **scenario pool** holds all the scenarios simulated for the given period, with annual available capactiy after disruption and customer demands.

In [8]:
SCENARIO_INDEX = 1 
# 2nd macro scenario from Klibi chapter 10/11 : 
# index 1 corresponds to 2nd scenario, that is, moderate steady growth
# most realistic business-as-usual trajectory

In [9]:
# build scenario pool

all_scenarios = [f"s_{i}" for i in range(1, N_SCENARIOS + 1)]
S = {}
prices_scenario_dict = {}

first_sc = data_SN["scenarios"][SCENARIO_INDEX]

for s_id in all_scenarios:
  sc_capacity = {}
  sc_demand = {}

  for t in periods:
    # available throughput capacity = nominal minus downtime loss
    for w in candidate_dcs:
      days_off = downtime_map.get((s_id, t, w), 0.0)
      nom_cap = CANDIDATE_WAREHOUSES[w]["nominal_cap"]

      sc_capacity[(w, t)] = compute_scenario_capacity(
          nominal_capacity=nom_cap,
          days_closed=days_off,
          operating_days_year=DAYS_PER_YEAR,
      )

    # benchmark
    for l in demand_zones:
      sc_demand[(l, t)] = first_sc["demand"][str(t)][l]
      prices_scenario_dict[(l, t, s_id)] = first_sc["prices"][str(t)][l]

  S[s_id] = {"omega": s_id, "capacity": sc_capacity, "demand": sc_demand}

print(f"Master Scenario Pool S built with {len(S)} scenarios.")

Master Scenario Pool S built with 100 scenarios.


## Optimization

We run optimization in three scenarios:
1. **The deterministic baseline** (no hazards taken into account): This would correspond to historical management of uncertainty and risk, where disasters will incur severe repair costs. It serves as baseline.
2. **The deterministic model** (average hazard scenario): we average parameters (here, capacities) over the entire pool of generated scenarios, and solve the model for this expected scenario.
3. **The stochastic model** finds a single static configuration across a sample of `20` disruption scenarios.


In [10]:
N_SCENARIOS = 20
N_OOS = 80

MIN_DEMAND_RATIO = 0.0

In [11]:
network_params = {
    "fixed_costs": data_SN["fixed_costs"],
    "prices": prices_scenario_dict,
    "flow_costs": flow_costs,
    "recourse_costs": data_SN["recourse_costs"],
    "penalty_unmet": {(l, t): 50.0 for l in demand_zones for t in periods},
    "max_recourse_ratio": data_SN["max_recourse_ratio"],
    "nominal_capacities": data_SN["nominal_capacities"],
    "min_deliveries": min_deliveries,
    "min_demand_ratio": MIN_DEMAND_RATIO,
}

In [ ]:
print("--- 1. Deterministic Model (No Hazards) ---")

nominal_scenario = [{
    "omega": "nominal",
    "capacity": {(w, t): CANDIDATE_WAREHOUSES[w]["nominal_cap"] for w in candidate_dcs for t in periods},
    "demand": {(l, t): first_sc["demand"][str(t)][l] for l in demand_zones for t in periods}
}]
prices_nom_dict = {(l, t, "nominal"): first_sc["prices"][str(t)][l] for l in demand_zones for t in periods}

results_naive = build_and_solve_one_stage(
    scenarios_sample=nominal_scenario,
    candidate_dcs=candidate_dcs,
    demand_zones=demand_zones,
    periods=periods,
    fixed_costs=network_params["fixed_costs"],
    prices=prices_nom_dict,
    flow_costs=network_params["flow_costs"],
    nominal_capacities=network_params["nominal_capacities"],
    min_demand_ratio=network_params["min_demand_ratio"],
    min_deliveries=network_params["min_deliveries"],
)

opened_dcs_naive = [w for w, is_open in results_naive["open_sites"].items() if is_open == 1]
obj_val_naive = results_naive["objective"]

fixed_cost_naive = sum(network_params["fixed_costs"][w] for w in opened_dcs_naive)
print(f"Open Facilities:    {set(opened_dcs_naive)}")
print(f"Fixed Network Cost: €{fixed_cost_naive:,.2f}")
print(f"Expected profit:    €{obj_val_naive:,.2f}\n")

--- 1. Deterministic Model (No Hazards) ---
Solver Status: Optimal
Open Facilities:    {'Dusseldorf', 'Turin'}
Fixed Network Cost: €15,000,000.00
Expected profit:    €19,319,625.00



In [ ]:
print("--- 2. Deterministic Model (Average Hazard Scenario) ---")

ev_sample_ids = list(S.keys())[:N_SCENARIOS]
ev_sample = [S[s_id] for s_id in ev_sample_ids]

# Average capacity and demand across scenarios
avg_capacity = {(w, t): np.mean([sc["capacity"][(w, t)] for sc in ev_sample]) for w in candidate_dcs for t in periods}
avg_demand = {(l, t): np.mean([sc["demand"][(l, t)] for sc in ev_sample]) for l in demand_zones for t in periods}

mean_scenario = [{"omega": "mean_ev_scenario", "capacity": avg_capacity, "demand": avg_demand}]
prices_ev_dict = {(l, t, "mean_ev_scenario"): np.mean([prices_scenario_dict[(l, t, sc["omega"])] for sc in ev_sample]) for l in demand_zones for t in periods}

results_ev = build_and_solve_one_stage(
    scenarios_sample=mean_scenario,
    candidate_dcs=candidate_dcs,
    demand_zones=demand_zones,
    periods=periods,
    fixed_costs=network_params["fixed_costs"],
    prices=prices_ev_dict,
    flow_costs=network_params["flow_costs"],
    nominal_capacities=network_params["nominal_capacities"],
    min_demand_ratio=network_params["min_demand_ratio"],
    min_deliveries=network_params["min_deliveries"],
)

opened_dcs_ev = [w for w, is_open in results_ev["open_sites"].items() if is_open == 1]
obj_val_ev = results_ev["objective"]

fixed_cost_ev = sum(network_params["fixed_costs"][w] for w in opened_dcs_ev)
print(f"Open Facilities:    {set(opened_dcs_ev)}")
print(f"Fixed Network Cost: €{fixed_cost_ev:,.2f}")
print(f"Expected profit:    €{obj_val_ev:,.2f}\n")

--- 2. Deterministic Model (Average Hazard Scenario) ---
Solver Status: Optimal
Open Facilities:    {'Dusseldorf', 'Turin'}
Fixed Network Cost: €15,000,000.00
Expected profit:    €18,732,391.66



In [ ]:
print("--- 3. Stochastic Model (One-Stage) ---")

one_stage_sample = ev_sample # using the same 20 scenarios

results_1stage = build_and_solve_one_stage(
    scenarios_sample=one_stage_sample,
    candidate_dcs=candidate_dcs,
    demand_zones=demand_zones,
    periods=periods,
    fixed_costs=network_params["fixed_costs"],
    prices=network_params["prices"],
    flow_costs=network_params["flow_costs"],
    nominal_capacities=network_params["nominal_capacities"],
    min_demand_ratio=network_params["min_demand_ratio"],
    min_deliveries=network_params["min_deliveries"],
)

opened_dcs_1stage = [w for w, is_open in results_1stage["open_sites"].items() if is_open == 1]
obj_val_1stage = results_1stage["objective"]

fixed_cost_1stage = sum(network_params["fixed_costs"][w] for w in opened_dcs_1stage)
print(f"Open Facilities:    {set(opened_dcs_1stage)}")
print(f"Fixed Network Cost: €{fixed_cost_1stage:,.2f}")
print(f"Expected profit:    €{obj_val_1stage:,.2f}\n")

--- 3. Stochastic Model (One-Stage) ---
Solver Status: Optimal
Open Facilities:    {'Toulouse', 'Dusseldorf', 'Turin'}
Fixed Network Cost: €25,500,000.00
Expected profit:    €10,360,580.14



## Out-of-Sample evaluation

We lock the network configurations and the planned static flows derived from the three models above.
We test and assess the three different model outcomes on data that was completely excluded during optimization, to see how each approach performs.

In [15]:
# Helper function to extract planned static flows F* for a fixed Y configuration
# Source: This function was written by Gemini

def get_static_flows_for_design(open_dcs, sample, price_dict):
    model = pulp.LpProblem("Extract_Flows", pulp.LpMaximize)
    F = {(w, l, t): pulp.LpVariable(f"F_{w}_{l}_{t}", lowBound=0) for t in periods for w in open_dcs for l in demand_zones}
    
    fixed_inv = sum(network_params["fixed_costs"][w] for w in open_dcs)
    scenario_objectives = []
    
    for sc in sample:
        om = sc["omega"]
        cashflow = []
        for t in periods:
            margin = pulp.lpSum((price_dict[(l, t, om)] - network_params["flow_costs"][(w, l)]) * F[(w, l, t)] for w in open_dcs for l in demand_zones)
            cashflow.append(margin)
        scenario_objectives.append(pulp.lpSum(cashflow))
        
    model += (1.0 / len(sample)) * pulp.lpSum(scenario_objectives) - fixed_inv
    
    for sc in sample:
        for t in periods:
            for l in demand_zones:
                d_val = sc["demand"][(l, t)]
                model += pulp.lpSum(F[(w, l, t)] for w in open_dcs) <= d_val
                model += pulp.lpSum(F[(w, l, t)] for w in open_dcs) >= network_params["min_demand_ratio"] * d_val
            for w in open_dcs:
                model += pulp.lpSum(F[(w, l, t)] for l in demand_zones) <= sc["capacity"][(w, t)]
                
    model.solve(pulp.COIN_CMD(path = "/opt/homebrew/bin/cbc", msg = False))
    return {(w, l, t): pulp.value(F[(w, l, t)]) for t in periods for w in open_dcs for l in demand_zones if pulp.value(F[(w, l, t)]) > 1e-4}

# Extract static flows for the 3 candidates
flows_naive = get_static_flows_for_design(opened_dcs_naive, nominal_scenario, prices_nom_dict)
flows_ev = get_static_flows_for_design(opened_dcs_ev, mean_scenario, prices_ev_dict)
flows_stoch = get_static_flows_for_design(opened_dcs_1stage, one_stage_sample, network_params["prices"])

def evaluate_static_plan(open_dcs, static_flows, eval_scenarios):
    fixed_inv = sum(network_params["fixed_costs"][w] for w in open_dcs)
    scenario_profits = []
    failures = 0

    for sc in eval_scenarios:
        cashflow = 0.0
        scenario_failed = False
        
        for t in periods:
            for w in open_dcs:
                total_attempted = sum(static_flows.get((w, l, t), 0.0) for l in demand_zones)
                cap_limit = sc["capacity"][(w, t)]
                
                scaling = min(1.0, cap_limit / total_attempted) if total_attempted > 0 else 1.0
                if scaling < 0.9999:
                    scenario_failed = True  # Network failed due to capacity breach
                    
                for l in demand_zones:
                    delivered_qty = static_flows.get((w, l, t), 0.0) * scaling
                    margin = (network_params["prices"][(l, t, sc["omega"])] - network_params["flow_costs"][(w, l)]) * delivered_qty
                    cashflow += margin

        scenario_profits.append(cashflow)
        if scenario_failed:
            failures += 1

    expected_profit = np.mean(scenario_profits) - fixed_inv
    failure_rate = (failures / len(eval_scenarios)) * 100.0
    return expected_profit, failure_rate

# Run Evaluation
eval_sample_ids = list(S.keys())[20:100]
eval_scenarios = [S[s_id] for s_id in eval_sample_ids]

candidates_to_test = {
    "1. Deterministic (No Hazards)": (opened_dcs_naive, flows_naive),
    "2. Deterministic (Average Hazard)": (opened_dcs_ev, flows_ev),
    "3. Stochastic (One-Stage)": (opened_dcs_1stage, flows_stoch)
}

print(f"--- Out-of-Sample Evaluation ({len(eval_scenarios)} Scenarios) ---")
for label, (dcs, flows) in candidates_to_test.items():
    exp_prof, fail_rate = evaluate_static_plan(dcs, flows, eval_scenarios)
    print(f"\n{label}")
    print(f"  Opened Sites:  {set(dcs)}")
    print(f"  Failure Rate:  {fail_rate:.1f}% (scenarios with capacity breach)")
    print(f"  Expected Cost (Net Profit): €{exp_prof:,.2f}")

--- Out-of-Sample Evaluation (80 Scenarios) ---

1. Deterministic (No Hazards)
  Opened Sites:  {'Dusseldorf', 'Turin'}
  Failure Rate:  91.2% (scenarios with capacity breach)
  Expected Cost (Net Profit): €17,468,995.61

2. Deterministic (Average Hazard)
  Opened Sites:  {'Dusseldorf', 'Turin'}
  Failure Rate:  87.5% (scenarios with capacity breach)
  Expected Cost (Net Profit): €17,226,454.11

3. Stochastic (One-Stage)
  Opened Sites:  {'Toulouse', 'Dusseldorf', 'Turin'}
  Failure Rate:  26.2% (scenarios with capacity breach)
  Expected Cost (Net Profit): €10,176,114.15


## Post-midterm modifications

The stochastic model is very conservative, because it must satisfy all constraints from the generated scenarios. However, if the scenario-pool contains a lot of extreme, this will lead to network configuration assuming the absolute worst, which is not realistic.

Following discussions during the midterm evaluation, we are generating more scenarios but cutting-off the extreme events to incorporate more realistic assumptions in the network under uncertainty design (there's no use to prepare the newtork for the absolute worst).

We cutting of 10% of the worst scenarios. 

In [17]:
NUMBER_TRUNCATION = 12 # we add it to the previous round, and we remove the 12 worse
NEW_SEED = SEED + 100

df_event_scenarios_extra = generate_mc_scenario_only(
    nb_n_monthly = NB_N_MONTHLY,
    n_years = N_YEARS,
    nb_p = NB_P_MONTHLY,
    country_weights = COUNTRY_WEIGHTS,
    duration_pools = DURATIONS,
    num_scenarios = NUMBER_TRUNCATION,
    severity_function = "severity_proxy",
    seed = NEW_SEED
)

df_facility_hits_extra = is_warehouse_hit(CANDIDATE_WAREHOUSES, df_event_scenarios_extra, default_alpha = ALPHA)
df_actual_hits_extra = df_facility_hits_extra[df_facility_hits_extra["is_hit"] == True].copy()

# actuals hits - we use attenuation probabilities 
df_actual_hits_extra["days_closed"] = df_actual_hits_extra.apply(
    lambda r: evaluate_proxies_continuous_Klibi(r["duration_days"], r["severity"], facility_type="depot")["days_closed_ttr"],
    axis=1
)

print(f"Generated {len(df_actual_hits_extra)} direct warehouse hits across scenarios.")

# AGGREGATE DOWNTIME AND SCENARIO CAPACITIES
df_scenario_impact_extra = df_actual_hits_extra.groupby(["scenario_id", "year", "warehouse"]).agg(
    total_days_closed = ("days_closed", "sum"),
    hit_count = ("is_hit", "sum"),
).reset_index()

downtime_map_extra = df_scenario_impact_extra.set_index(["scenario_id", "year", "warehouse"])["total_days_closed"].to_dict()


Generated 84 direct warehouse hits across scenarios.


In [18]:
S_extra = {}
first_sc = data_SN["scenarios"][SCENARIO_INDEX]

for i in range(1, N_SCENARIOS + 1):

  s_id_string = f"s_extra_{i}"
  sc_capacity = {}
  sc_demand = {}

  for t in periods:
    for w in candidate_dcs:
      days_off = downtime_map_extra.get((i, t, w), 0.0)
      nom_cap = CANDIDATE_WAREHOUSES[w]["nominal_cap"]

      sc_capacity[(w, t)] = compute_scenario_capacity(
          nominal_capacity=nom_cap,
          days_closed=days_off,
          operating_days_year=DAYS_PER_YEAR,
      )

    # benchmark
    for l in demand_zones:
      sc_demand[(l, t)] = first_sc["demand"][str(t)][l]
      prices_scenario_dict[(l, t, s_id_string)] = first_sc["prices"][str(t)][l]

  S_extra[s_id_string] = {"omega": s_id_string, "capacity": sc_capacity, "demand": sc_demand}

print(f"Master Scenario Pool S built with {len(S_extra)} scenarios.")

Master Scenario Pool S built with 20 scenarios.


In [20]:
# AGGREGATE
S_combined = {**S, **S_extra}
print(f"Combined pool size: {len(S_combined)} scenarios.")

scenario_capacities = []
for s_id, sc_data in S_combined.items() :
    total_cap = sum(sc_data["capacity"].values())
    scenario_capacities.append((s_id, total_cap))

scenario_capacities.sort(key = lambda x: x[1])

worst_12_scenarios = [item[0] for item in scenario_capacities[:NUMBER_TRUNCATION]]
print(f"Removing the 12 worst scenarios: {worst_12_scenarios}")

S_final = {k: v for k, v in S_combined.items() if k not in worst_12_scenarios}
print(f"Final Master Scenario Pool ready with {len(S_final)} scenarios.")

Combined pool size: 120 scenarios.
Removing the 12 worst scenarios: ['s_93', 's_2', 's_22', 's_32', 's_82', 's_47', 's_21', 's_28', 's_95', 's_52', 's_42', 's_17']
Final Master Scenario Pool ready with 108 scenarios.


In [ ]:
# OPTIMIZATION
# no need to re-do for the first model since not stochastic

print("--- 2. Deterministic Model (Average Hazard Scenario) ---")

N_SCENARIO_POOL = 20

ev_sample_ids = list(S_final.keys())[:N_SCENARIO_POOL]
ev_sample = [S_final[s_id] for s_id in ev_sample_ids]

combined_prices = {**network_params["prices"], **prices_scenario_dict}

# Average capacity and demand across scenarios
avg_capacity = {(w, t): np.mean([sc["capacity"][(w, t)] for sc in ev_sample]) for w in candidate_dcs for t in periods}
avg_demand = {(l, t): np.mean([sc["demand"][(l, t)] for sc in ev_sample]) for l in demand_zones for t in periods}

mean_scenario = [{"omega": "mean_ev_scenario", "capacity": avg_capacity, "demand": avg_demand}]
prices_ev_dict = {(l, t, "mean_ev_scenario"): np.mean([combined_prices[(l, t, sc["omega"])] for sc in ev_sample]) for l in demand_zones for t in periods}

results_ev = build_and_solve_one_stage(
    scenarios_sample=mean_scenario,
    candidate_dcs=candidate_dcs,
    demand_zones=demand_zones,
    periods=periods,
    fixed_costs=network_params["fixed_costs"],
    prices=prices_ev_dict,
    flow_costs=network_params["flow_costs"],
    nominal_capacities=network_params["nominal_capacities"],
    min_demand_ratio=network_params["min_demand_ratio"],
    min_deliveries=network_params["min_deliveries"],
)

opened_dcs_ev = [w for w, is_open in results_ev["open_sites"].items() if is_open == 1]
obj_val_ev = results_ev["objective"]

fixed_cost_ev = sum(network_params["fixed_costs"][w] for w in opened_dcs_ev)
print(f"Open Facilities:    {set(opened_dcs_ev)}")
print(f"Fixed Network Cost: €{fixed_cost_ev:,.2f}")
print(f"Expected profit:    €{obj_val_ev:,.2f}\n")

print("--- 3. Stochastic Model (One-Stage) ---")

one_stage_sample = ev_sample # using the same 20 scenarios

results_1stage = build_and_solve_one_stage(
    scenarios_sample=one_stage_sample,
    candidate_dcs=candidate_dcs,
    demand_zones=demand_zones,
    periods=periods,
    fixed_costs=network_params["fixed_costs"],
    prices=combined_prices,
    flow_costs=network_params["flow_costs"],
    nominal_capacities=network_params["nominal_capacities"],
    min_demand_ratio=network_params["min_demand_ratio"],
    min_deliveries=network_params["min_deliveries"],
)

opened_dcs_1stage = [w for w, is_open in results_1stage["open_sites"].items() if is_open == 1]
obj_val_1stage = results_1stage["objective"]

fixed_cost_1stage = sum(network_params["fixed_costs"][w] for w in opened_dcs_1stage)
print(f"Open Facilities:    {set(opened_dcs_1stage)}")
print(f"Fixed Network Cost: €{fixed_cost_1stage:,.2f}")
print(f"Expected profit:    €{obj_val_1stage:,.2f}\n")

--- 2. Deterministic Model (Average Hazard Scenario) ---
Solver Status: Optimal
Open Facilities:    {'Dusseldorf', 'Turin'}
Fixed Network Cost: €15,000,000.00
Expected profit:    €18,899,174.55

--- 3. Stochastic Model (One-Stage) ---
Solver Status: Optimal
Open Facilities:    {'Toulouse', 'Dusseldorf', 'Turin'}
Fixed Network Cost: €25,500,000.00
Expected profit:    €11,483,234.50



In [22]:
# OUT OF SAMPLE WITH THESE CONFIGURATIONS

flows_naive = get_static_flows_for_design(opened_dcs_naive, nominal_scenario, prices_nom_dict)
flows_ev = get_static_flows_for_design(opened_dcs_ev, mean_scenario, prices_ev_dict)
flows_stoch = get_static_flows_for_design(opened_dcs_1stage, one_stage_sample, combined_prices)

network_params["prices"].update(prices_scenario_dict)

# Run Evaluation
eval_sample_ids = list(S_final.keys())[20:100]
eval_scenarios = [S_final[s_id] for s_id in eval_sample_ids]

candidates_to_test = {
    "1. Deterministic (No Hazards)": (opened_dcs_naive, flows_naive),
    "2. Deterministic (Average Hazard)": (opened_dcs_ev, flows_ev),
    "3. Stochastic (One-Stage)": (opened_dcs_1stage, flows_stoch)
}

print(f"--- Out-of-Sample Evaluation ({len(eval_scenarios)} Scenarios) ---")
for label, (dcs, flows) in candidates_to_test.items():
    exp_prof, fail_rate = evaluate_static_plan(dcs, flows, eval_scenarios)
    print(f"\n{label}")
    print(f"  Opened Sites:  {set(dcs)}")
    print(f"  Failure Rate:  {fail_rate:.1f}% (scenarios with capacity breach)")
    print(f"  Expected Cost (Net Profit): €{exp_prof:,.2f}")

--- Out-of-Sample Evaluation (80 Scenarios) ---

1. Deterministic (No Hazards)
  Opened Sites:  {'Dusseldorf', 'Turin'}
  Failure Rate:  76.2% (scenarios with capacity breach)
  Expected Cost (Net Profit): €17,984,979.24

2. Deterministic (Average Hazard)
  Opened Sites:  {'Dusseldorf', 'Turin'}
  Failure Rate:  71.2% (scenarios with capacity breach)
  Expected Cost (Net Profit): €17,796,100.23

3. Stochastic (One-Stage)
  Opened Sites:  {'Toulouse', 'Dusseldorf', 'Turin'}
  Failure Rate:  17.5% (scenarios with capacity breach)
  Expected Cost (Net Profit): €11,382,712.82


Lean network opening, opening Düsseldorf and Turin. While this footprint projects a massive expected net profit, but brittle: 83% failure rate, already better (but this is simply because we eliminate in the Out-Of-Sample the very worst-case scenarios). 

The stochastic model does recognize the structural redundancy, and opens Toulouse, absorbing the demand when other nodes fail and dropping network failure rate to 30%. It does have a higher failure rate but higher expected price as well, since we're now not optimizing against the very worst-cases.